# Interview Cheat Sheet — Tough Questions & Strong Answers

Complements `theory_guide.md`. Focus: questions an expert interviewer would ask to probe depth of understanding.

## Forward Modeling Questions

### "Why Transverse Magnetic z (TMz) and not Transverse Electric z (TEz)?"

TMz has $E_z$, $H_x$, $H_y$. The rebars are infinitely long cylinders perpendicular to the scan plane (into the page). For this geometry, the electric field polarized along the rebar axis ($E_z$) creates the strongest scattering response. TEz would have $H_z$ polarized along the rebar — weaker contrast because the magnetic permeability contrast is negligible ($\mu_r = 1$ everywhere). TMz is the standard choice for Ground-Penetrating Radar (GPR) line scans over parallel rebars.

### "Why not use a higher-order Finite-Difference Time-Domain (FDTD) scheme?"

Second-order Yee scheme is standard and sufficient at 16+ points per wavelength. Higher-order (4th-order in space) would allow coarser grids but: (1) complicates Perfectly Matched Layer (PML) implementation significantly, (2) wider stencil creates issues at material interfaces, (3) the grid is small enough that computation isn't the bottleneck. The trade-off doesn't pay off for this problem size.

### "What happens if you reduce grid points per wavelength to 5?"

Severe numerical dispersion — waves travel slower than they should in the discrete grid. The B-scan hyperbolas would shift to later arrival times, giving incorrect depth estimates. At 5 pts/wavelength, the dispersion error can exceed 5%. The Yee scheme needs 10+ for acceptable accuracy, and 15–20 for good accuracy.

### "Why is your time step 90% of the Courant–Friedrichs–Lewy (CFL) limit, not 99%?"

At exactly the CFL limit, numerical dispersion is zero along the grid axes but maximum along diagonals ($45°$). A slight reduction ($0.9$) provides a margin for rounding errors and reduces anisotropic dispersion. Going lower (e.g., $0.5$) wastes computation. $0.9$ is a well-established practical choice (Taflove & Hagness recommend $0.9$–$0.95$).

### "How does your PML handle evanescent waves?"

That's exactly why we use Complex Frequency Shifted Perfectly Matched Layer (CFS-PML). Standard PML absorbs propagating waves but can amplify evanescent/near-field waves, causing late-time instability. The CFS extension adds the $\alpha$ parameter ($\alpha_{\max} = 0.05$) which shifts the pole away from the origin in frequency domain, stabilizing evanescent wave absorption. The $\kappa$ parameter ($\kappa_{\max} = 5$) provides coordinate stretching that improves absorption of waves at grazing incidence.

### "Why a soft source instead of hard source?"

Hard source ($E_z = \text{value}$) creates an artificial Perfect Electric Conductor (PEC) point — waves reflect off the source location. Soft source ($E_z \mathrel{+}= \text{value}$) is equivalent to injecting a current density $J_z$, which radiates freely without creating spurious reflections. Critical for GPR simulation where the source is inside the computational domain, not on a boundary.

## Inversion Questions

### "Walk me through the adjoint gradient derivation"

1. Define the Lagrangian:
$$\mathcal{L} = J(\mathbf{d}) + \langle \boldsymbol{\lambda},\, \mathbf{A}\mathbf{u} - \mathbf{s} \rangle$$
where $\mathbf{A}$ is the wave operator, $\mathbf{u}$ is the field, and $\mathbf{s}$ is the source.

2. Set $\partial\mathcal{L}/\partial\mathbf{u} = 0$ to obtain the adjoint equation:
$$\mathbf{A}^{T} \boldsymbol{\lambda} = -\frac{\partial J}{\partial \mathbf{d}}$$
(data residual injected at receivers).

3. The adjoint equation is the same FDTD but run **backward in time** with time-reversed residual as source.

4. Then:
$$\frac{\partial J}{\partial m} = \left\langle \boldsymbol{\lambda},\, \frac{\partial \mathbf{A}}{\partial m}\, \mathbf{u} \right\rangle$$
which, for $\varepsilon_r$, gives the cross-correlation formula:
$$g = -\varepsilon_0 \sum_{t} \left( E_z^{\text{adj}} \cdot \frac{\partial E_z^{\text{fwd}}}{\partial t} \right) \Delta t$$

5. Key insight: this costs only **2 simulations per source** (forward + adjoint) regardless of parameter count.

### "Why normalize the gradient instead of using the physical magnitude?"

The raw adjoint gradient has magnitude $\sim\!10^{-17}$ because it is the product of two very small field quantities ($E_z$ values $\sim\!10^{-3}$) times $\varepsilon_0 \approx 8.85 \times 10^{-12}$ times $\Delta t \approx 4.24 \times 10^{-12}$. While physically correct, this magnitude is near machine precision, making finite-precision line search impossible. Normalizing to unit max preserves the **direction** (which determines WHERE to update) while letting Limited-memory Broyden–Fletcher–Goldfarb–Shanno with Bound constraints (L-BFGS-B)'s line search determine HOW MUCH to update. This is a standard practice in seismic Full-Waveform Inversion (FWI) (see Virieux & Operto 2009).

### "What are the limitations of your inversion?"

1. **Local minimum**: Gradient-based methods find local minima. If the initial model is too far from truth, inversion can converge to a wrong model. Mitigation: start from a reasonable prior (homogeneous concrete).
2. **Cycle-skipping**: If predicted and observed waveforms differ by more than half a cycle, the gradient points the wrong way. Mitigation: use low-frequency content first (multi-scale approach, not implemented here).
3. **Crosstalk**: We invert only $\varepsilon_r$ but conductivity ($\sigma$) also affects the data. Any $\varepsilon_r$ change that compensates for conductivity effects is a false positive. Mitigation: Total Variation (TV) regularization helps by penalizing smooth gradients.
4. **Resolution**: Limited by wavelength — cannot recover features smaller than $\sim\!\lambda/2$ (about 40 mm in concrete at 1.5 GHz). The rebars (12 mm) are detected by their impedance contrast, not geometrically resolved.

### "Why TV regularization specifically?"

TV (Total Variation) promotes piecewise-constant models with sharp interfaces. This is physically appropriate because the concrete/rebar boundary IS a sharp discontinuity — there's no smooth gradient between concrete ($\varepsilon_r = 6$) and steel ($\varepsilon_r \approx 1$). $L_2$ regularization would smooth this boundary, reducing recovery quality. TV is the right prior for this geometry.

### "How would you extend this to dual-parameter inversion ($\varepsilon_r$ AND $\sigma$)?"

1. Derive a second gradient formula for $\sigma$:
$$g_\sigma = -\sum_{t} \left( E_z^{\text{adj}} \cdot E_z^{\text{fwd}} \right) \Delta t$$
(different kernel — no time derivative).
2. Apply separate step sizes for each parameter (they have different sensitivities).
3. Use cross-gradient regularization to enforce structural similarity between $\varepsilon_r$ and $\sigma$ models.
4. Main challenge: ill-posedness increases — more unknowns with the same data. Would need more data (multiple frequencies, wider aperture) or stronger priors.

## GPU Questions

### "Why is your Graphics Processing Unit (GPU) slower at the project grid size?"

The project grid ($180 \times 280 = 50\text{K cells}$) is too small to saturate the GPU's parallelism. Each Compute Unified Device Architecture (CUDA) kernel launch has $\sim\!10$–$50\;\mu\text{s}$ overhead, and memory transfer latency dominates when there's not enough arithmetic to amortize it. Our scaling benchmark shows the crossover: GPU becomes faster at $\sim\!50\text{K}$ cells and reaches $7\times$ speedup at 800K cells. Real-world GPR problems (3D, fine grids) have millions of cells where GPU speedup is $30$–$100\times$.

### "Why CuPy instead of writing CUDA kernels?"

1. **Correctness first**: CuPy is a drop-in NumPy replacement — same code, same equations, easy to verify.
2. **Development speed**: Writing custom CUDA kernels for the FDTD stencil is straightforward but requires managing thread blocks, shared memory, boundary handling.
3. **Sufficient for this grid**: At our problem size, CuPy's overhead is negligible compared to custom kernels.
4. **Production path**: If we needed maximum performance, I'd write custom CUDA kernels for the stencil operations ($3\times3$ stencil maps cleanly to GPU shared memory tiles) and use PyTorch for automatic differentiation (autodiff)-based gradient computation.

### "What would you do differently for a production GPU implementation?"

1. **Port Convolutional Perfectly Matched Layer (CPML) to GPU**: Currently only field updates are on GPU. CPML corrections add $\sim\!10$–$20\%$ overhead.
2. **Fuse kernels**: Combine H-update + CPML into a single kernel to reduce memory traffic.
3. **Use shared memory**: Load 2D tile + halo into shared memory for the stencil operations.
4. **Checkpointing**: Instead of storing all forward fields (760 MB), use binomial checkpointing to reduce memory to $O(\sqrt{N})$ with modest recomputation cost.
5. **Multi-GPU**: Domain decomposition along the $x$-axis with halo exchange (1 cell overlap needed for the Yee stencil).

## Extension Questions

### "How would you go from 2D to 3D?"

1. Add the third field components: $E_x$, $E_y$, $H_z$ (full 6-component Maxwell's equations).
2. Grid becomes 3D: memory scales as $N^3$ instead of $N^2$.
3. PML needed on 6 faces instead of 4 edges.
4. Computation scales dramatically — GPU acceleration becomes essential.
5. Main challenge: memory for adjoint (storing all fields) — would need checkpointing.

### "How would you handle dispersive concrete?"

Real concrete is dispersive — permittivity and conductivity vary with frequency. Model using Auxiliary Differential Equations (ADE): add Debye or Cole–Cole relaxation terms to the E-field update. Each pole adds one auxiliary variable per cell. For GPR in concrete, a single Debye pole is usually sufficient to capture the frequency dependence across the 0.5–3 GHz band.

### "What about antenna modeling?"

Our point source is an idealization. Real GPR antennas have:
1. Finite size (bow-tie pattern, $\sim\!10$ cm)
2. Radiation pattern (not omnidirectional)
3. Feed impedance and matching
4. Near-field coupling with the ground

For more realistic simulation: model the antenna geometry explicitly (metallic arms + feed gap) or use a measured/computed antenna transfer function convolved with the source wavelet. gprMax uses explicit antenna modeling.

## DGX Spark Specific

### Hardware specs to mention:
- NVIDIA GB10 GPU
- 128 GB unified memory (no separate CPU/GPU memory — simplifies large-problem deployment)
- CUDA 13.0
- Advanced RISC Machines (ARM) architecture (aarch64)

### Results to show:
- Forward B-scan: 3 clear rebar hyperbolas
- Inversion: rebar recovery from blind start
- GPU scaling: $3$–$7\times$ speedup (would be $30$–$100\times$ for production-size 3D problems)
- All tests passing (6/6)

## Appendix: Comprehensive Glossary

### FDTD Concepts

**FDTD (Finite-Difference Time-Domain)**
A numerical method for solving Maxwell's equations directly in the time domain. Space is discretised onto a grid and time is advanced in discrete steps. At each step, finite-difference approximations replace the spatial and temporal derivatives in Maxwell's curl equations, producing explicit update equations that march the electric and magnetic fields forward. FDTD is widely used because it handles broadband sources in a single run, naturally accommodates heterogeneous materials, and is straightforward to implement and parallelise.

**Yee Grid (Yee Cell)**
The staggered spatial arrangement introduced by Kane Yee in 1966 in which electric and magnetic field components are offset by half a grid cell in both space and time. In 2D TMz mode, $E_z$ lives at integer grid points $(i,j)$ while $H_x$ and $H_y$ live at half-integer offsets. This staggering is not arbitrary: it ensures that every finite-difference curl calculation uses the field values that are naturally centred around the point of interest, yielding second-order accuracy without any averaging or interpolation. The Yee cell is the foundation that makes FDTD both simple and accurate.

**Leapfrog Time-Stepping**
The temporal counterpart of the Yee spatial staggering. Electric fields are computed at integer time steps ($n$) and magnetic fields at half-integer time steps ($n+\tfrac{1}{2}$), so they "leapfrog" over each other in time. Each field update uses the most recently computed value of the other field, making the scheme fully explicit (no matrix solves needed). Leapfrog is second-order accurate in time and, combined with the Yee grid, produces a symplectic-like integrator that conserves a discrete energy and remains stable as long as the CFL condition is satisfied.

**Stencil**
The pattern of neighbouring grid points used to approximate a spatial derivative via finite differences. In the standard second-order Yee scheme the stencil is compact: each derivative uses only the two nearest neighbours along that axis (a 3-point stencil per dimension). Higher-order FDTD schemes widen the stencil (e.g., 5 or 7 points) to reduce numerical dispersion at the cost of more complex boundary handling and PML formulations. The stencil width also determines the minimum halo size needed for domain decomposition in parallel implementations.

**CFL Condition (Courant-Friedrichs-Lewy)**
The stability criterion that imposes a maximum allowable time step for the explicit FDTD scheme. In 2D it reads $c\,\Delta t \le \bigl(1/\Delta x^2 + 1/\Delta y^2\bigr)^{-1/2}$, where $c$ is the maximum wave speed in the domain. Violating the CFL limit causes exponential growth of field values and immediate simulation blow-up. In practice the time step is set to 90-95% of the CFL limit to provide a safety margin against rounding errors and to reduce the anisotropy of numerical dispersion.

**Numerical Dispersion**
An artefact of spatial and temporal discretisation in which the phase velocity of a wave in the discrete grid differs from its true physical value. The error depends on the propagation direction relative to the grid axes, the number of grid points per wavelength, and the Courant number. At 10 points per wavelength the phase-velocity error can be several percent; at 20 points it drops below 0.1%. Numerical dispersion causes pulse broadening and arrival-time errors, directly degrading the accuracy of GPR simulations and subsequent inversions.

**Grid Points per Wavelength**
The number of spatial cells that span one minimum wavelength $\lambda_{\min}$ in the simulation. This is the primary control on numerical dispersion: the Yee scheme typically requires at least 10 points per wavelength for acceptable accuracy and 15-20 for high-fidelity results. The minimum wavelength is set by the highest significant frequency in the source spectrum and the lowest wave speed (highest $\varepsilon_r$) in the model. Under-sampling leads to severe dispersion, phase errors, and ultimately wrong inversion results.

**Courant Number**
The dimensionless ratio $S = c\,\Delta t / \Delta x$ (or its multi-dimensional generalisation). It measures how far a wave travels in one time step relative to the grid spacing. The CFL condition requires $S \le 1/\sqrt{d}$ in $d$ dimensions. Choosing $S$ close to its maximum minimises the number of time steps but maximises anisotropic dispersion. The practical sweet spot is $S \approx 0.9 / \sqrt{d}$, which balances efficiency against dispersion uniformity.

**Explicit vs. Implicit Schemes**
FDTD is an explicit scheme: each new field value is computed directly from known values at previous time steps, with no need to solve a system of equations. This makes it fast and easy to parallelise but imposes the CFL stability limit. Implicit schemes (e.g., ADI-FDTD) allow larger time steps by solving coupled equations at each step, but the matrix solves are expensive and harder to parallelise. For GPR-scale problems, explicit FDTD is almost always preferred.

---

### Electromagnetic Concepts

**Maxwell's Equations**
The four fundamental partial differential equations governing all classical electromagnetic phenomena. In FDTD for GPR we use the two curl equations: Faraday's law ($\nabla \times \mathbf{E} = -\mu\,\partial\mathbf{H}/\partial t$) and Ampere's law ($\nabla \times \mathbf{H} = \varepsilon\,\partial\mathbf{E}/\partial t + \sigma\mathbf{E} + \mathbf{J}$). The divergence equations ($\nabla \cdot \mathbf{D} = \rho$ and $\nabla \cdot \mathbf{B} = 0$) are automatically satisfied if the initial conditions satisfy them and the curl equations are solved consistently — a property the Yee scheme preserves by construction.

**Permittivity ($\varepsilon$)**
A material property that quantifies how strongly a material polarises in response to an applied electric field, thereby affecting the speed and wavelength of electromagnetic waves. It is usually expressed as $\varepsilon = \varepsilon_0\,\varepsilon_r$, where $\varepsilon_0 \approx 8.854 \times 10^{-12}$ F/m is the vacuum permittivity and $\varepsilon_r$ is the dimensionless relative permittivity. In GPR, permittivity is the primary parameter of interest: concrete has $\varepsilon_r \approx 4$-$8$, water $\approx 80$, air $= 1$. Contrasts in $\varepsilon_r$ produce the reflections that form a GPR image.

**Relative Permittivity ($\varepsilon_r$)**
The ratio of a material's permittivity to the vacuum permittivity $\varepsilon_0$. It directly controls the wave speed via $v = c / \sqrt{\varepsilon_r}$ (for non-magnetic, low-loss materials). This is the parameter recovered by the full-waveform inversion in this project. Mapping $\varepsilon_r$ reveals subsurface structure because different materials (air, concrete, rebar, soil, water) have distinct $\varepsilon_r$ values that create reflection boundaries.

**Conductivity ($\sigma$)**
A material property (in S/m) that governs ohmic losses — the conversion of electromagnetic energy into heat as currents flow through a resistive medium. In the FDTD $E$-field update, conductivity appears as an exponential damping factor $e^{-\sigma\,\Delta t/(2\varepsilon)}$ that attenuates the field at every time step. Higher conductivity means stronger signal attenuation and reduced GPR penetration depth. In concrete, $\sigma$ is typically 0.01-0.1 S/m; in wet clay it can exceed 1 S/m, making GPR impractical.

**Permeability ($\mu$)**
A material property that quantifies the magnetic response of a medium. Almost all materials encountered in GPR work (concrete, soil, air, water, rebar) are non-magnetic, meaning $\mu_r = 1$ and $\mu = \mu_0 = 4\pi \times 10^{-7}$ H/m. This is why GPR inversion typically targets $\varepsilon_r$ and $\sigma$ only. Magnetic materials (e.g., magnetite-rich soils) are the exception and would require a modified inversion scheme.

**Wave Impedance**
The ratio of the electric field amplitude to the magnetic field amplitude in a plane wave, given by $\eta = \sqrt{\mu / \varepsilon}$ for lossless media. In free space, $\eta_0 \approx 377\;\Omega$. The reflection coefficient at a planar interface between two media is determined by the impedance contrast: $R = (\eta_2 - \eta_1)/(\eta_2 + \eta_1)$. Larger impedance contrasts produce stronger reflections — this is why metal objects (effectively $\eta \to 0$) are strong GPR reflectors.

**Reflection Coefficient**
The ratio of the reflected electric field amplitude to the incident amplitude at a planar interface. For normal incidence between two non-magnetic media, $R = (\sqrt{\varepsilon_{r1}} - \sqrt{\varepsilon_{r2}}) / (\sqrt{\varepsilon_{r1}} + \sqrt{\varepsilon_{r2}})$. The sign indicates the phase: a wave going from low to high permittivity experiences a phase reversal. The magnitude determines how much energy is reflected versus transmitted, directly controlling the brightness of reflectors in a GPR B-scan.

**Skin Depth**
The distance at which an electromagnetic wave's amplitude decays to $1/e$ ($\approx 37\%$) of its surface value in a conductive medium. For a good dielectric with moderate loss, $\delta \approx 1/(\sigma\sqrt{\mu/(4\varepsilon)}\,)$ at a given frequency. Skin depth sets the practical penetration limit of GPR: signals must travel to a target and back, so the maximum detectable depth is roughly $2$-$3$ skin depths. Higher frequencies and more conductive media both reduce the skin depth.

**Polarisation (TMz / TEz)**
In 2D FDTD, the six-component Maxwell's equations decouple into two independent three-component sets. TMz (Transverse Magnetic to $z$) has field components $E_z$, $H_x$, $H_y$ — the electric field is polarised along the $z$-axis. TEz (Transverse Electric to $z$) has $H_z$, $E_x$, $E_y$. For GPR scanning over elongated targets like rebars (running parallel to $z$), TMz produces the strongest scattering response because the electric field is aligned with the target axis, inducing large currents.

**Wavelength ($\lambda$)**
The spatial period of an electromagnetic wave, related to frequency by $\lambda = v / f$ where $v = c/\sqrt{\varepsilon_r}$ is the wave speed in the medium. In concrete ($\varepsilon_r \approx 6$) at 1.5 GHz, $\lambda \approx 82$ mm. The wavelength sets the fundamental resolution limit of GPR imaging ($\sim\!\lambda/2$) and determines the required grid spacing for FDTD simulation (typically $\lambda/15$ to $\lambda/20$).

---

### GPR Concepts

**GPR (Ground-Penetrating Radar)**
A non-destructive geophysical technique that transmits short pulses of electromagnetic energy into a medium (ground, concrete, etc.) and records the reflections from subsurface interfaces and objects. GPR operates in the MHz to GHz range: higher frequencies give better resolution but lower penetration. Applications include rebar detection in concrete, utility mapping, archaeology, and ice-thickness measurement.

**A-scan**
A single time-domain trace recorded at one transmitter-receiver position, showing the received signal amplitude as a function of two-way travel time. Each A-scan is a 1D record: the horizontal axis is time (convertible to depth if the wave speed is known) and the vertical axis is signal amplitude. Reflections from subsurface interfaces appear as wavelets in the A-scan, with arrival time proportional to depth and amplitude proportional to the impedance contrast.

**B-scan (Radargram)**
A 2D GPR image formed by juxtaposing many A-scans recorded at successive positions along a survey line. The horizontal axis is antenna position, the vertical axis is two-way travel time (or depth), and pixel intensity represents signal amplitude. B-scans are the primary data product of GPR surveys. Point scatterers (like rebars viewed in cross-section) produce characteristic hyperbolic signatures in B-scans due to the varying source-scatterer distance as the antenna moves.

**Hyperbola**
The distinctive inverted-U shape that a point scatterer or a small cylindrical object (like a rebar cross-section) produces in a B-scan. As the antenna moves along the survey line, the travel time to the scatterer first decreases (approaching), reaches a minimum (directly above), then increases (receding), tracing a hyperbolic curve. The opening angle of the hyperbola depends on the wave velocity in the host medium — wider for faster media. Hyperbola fitting is a classical method for estimating subsurface wave velocity and target depth.

**Direct Wave**
The electromagnetic pulse that travels directly from the transmitter to the receiver through the air or along the surface, without reflecting from any subsurface feature. It arrives first (before any reflections) and is typically the strongest event in the A-scan. In GPR processing and inversion, the direct wave is often removed because it obscures shallow reflections and does not carry subsurface information. Its arrival time can be used to verify the transmitter-receiver separation.

**Surface Reflection**
The reflection generated at the air-ground (or air-concrete) interface due to the permittivity contrast. It arrives shortly after the direct wave and can be very strong (e.g., $R \approx 0.42$ for an air-concrete interface with $\varepsilon_r = 6$). In the B-scan it appears as a near-horizontal band at early times. Like the direct wave, it is often removed before inversion to allow the algorithm to focus on deeper features.

**Two-Way Travel Time (TWT)**
The total time for a radar pulse to travel from the transmitter down to a reflector and back up to the receiver. Depth $d$ is related to TWT by $d = v \cdot \text{TWT} / 2$, where $v$ is the wave speed in the medium. Converting TWT to depth requires knowledge of $\varepsilon_r$, which is often the unknown that motivates inversion in the first place.

**Migration**
A signal-processing technique that collapses the diffraction hyperbolas in a B-scan back to their point-source locations, producing a focused image of the subsurface. Migration requires an estimate of the velocity model. It is conceptually related to inversion but operates on a simpler imaging principle (wavefield back-propagation) without iteratively fitting waveforms. FWI can be viewed as a much more rigorous alternative to migration.

---

### Inversion Concepts

**Forward Problem**
Given a known model of the subsurface (permittivity, conductivity, geometry), compute the synthetic data (B-scan) that a GPR system would record. In this project, the forward problem is solved by running an FDTD simulation. The forward problem has a unique solution: one model produces one dataset.

**Inverse Problem**
Given observed GPR data, determine the subsurface model that produced it. This is fundamentally harder than the forward problem because it is non-unique (many models can fit the data) and ill-posed (small data perturbations can cause large model changes). Regularisation and prior information are needed to constrain the solution.

**FWI (Full-Waveform Inversion)**
An iterative optimisation technique that minimises the misfit between observed and synthetic GPR data by adjusting the subsurface model. Unlike ray-based or migration methods, FWI uses the complete waveform — amplitude, phase, and all arrivals — to extract maximum information from the data. Each iteration requires a forward simulation and an adjoint simulation, and the model is updated along the gradient direction. FWI can achieve resolution beyond the classical diffraction limit but is computationally expensive and susceptible to local minima.

**Misfit Function (Objective Function, Cost Function)**
A scalar measure of the discrepancy between observed and simulated data, typically the $L_2$ norm of the data residual: $J = \tfrac{1}{2}\sum_{r,t} \|d^{\text{obs}}_{r,t} - d^{\text{syn}}_{r,t}\|^2$. The goal of inversion is to minimise this function. Other misfit measures exist (envelope, cross-correlation, Wasserstein distance) that can improve convergence in the presence of cycle-skipping, but $L_2$ is the most common starting point.

**Adjoint Method**
A mathematical technique for computing the gradient of the misfit function with respect to all model parameters simultaneously, using only two simulations per source: one forward and one adjoint. The adjoint simulation solves the same wave equation but backward in time, with the data residual injected at receiver locations as a source. The gradient is then obtained by cross-correlating the forward and adjoint fields. Without the adjoint method, computing the gradient would require one simulation per parameter — millions for a typical model.

**Adjoint Source**
The source term used in the adjoint (backward) simulation. For an $L_2$ misfit, the adjoint source at each receiver is simply the time-reversed data residual: $d^{\text{obs}} - d^{\text{syn}}$, reversed in time and injected at the receiver location. The adjoint source encodes where and when the forward simulation disagrees with the observations, guiding the gradient to correct the model in the right places.

**Gradient (Sensitivity Kernel)**
The derivative of the misfit function with respect to each model parameter, computed via the adjoint method. For relative permittivity, the gradient at each grid cell is $g = -\varepsilon_0 \sum_t E_z^{\text{adj}} \cdot \partial_t E_z^{\text{fwd}} \cdot \Delta t$. The gradient indicates the direction of steepest descent in model space: negative values indicate that increasing $\varepsilon_r$ at that location would reduce the misfit. The gradient map highlights which regions of the model the data are most sensitive to.

**Hessian**
The matrix of second derivatives of the misfit function with respect to model parameters. It encodes the curvature of the misfit landscape and the trade-offs between parameters. The full Hessian is prohibitively expensive to compute ($N^2$ entries for $N$ parameters), but approximate Hessians (e.g., the Gauss-Newton or L-BFGS approximation) provide crucial scaling information that accelerates convergence and improves resolution. The diagonal of the Hessian acts as a natural preconditioner for the gradient.

**L-BFGS-B (Limited-memory BFGS with Bound Constraints)**
A quasi-Newton optimisation algorithm that approximates the inverse Hessian using a limited history of gradient differences (typically 5-20 past iterations). The "B" indicates support for box constraints on parameters (e.g., $1 \le \varepsilon_r \le 15$). L-BFGS-B is the workhorse of large-scale FWI because it provides superlinear convergence without storing or inverting the full Hessian matrix. It requires only the misfit value and gradient at each iteration.

**Line Search**
A 1D optimisation sub-problem performed at each iteration to determine the optimal step size along the current search direction. Given a descent direction (from the gradient or quasi-Newton update), the line search evaluates the misfit at a few trial step sizes to find one that satisfies sufficient decrease conditions (e.g., Wolfe conditions). A good line search prevents overshooting and ensures monotonic decrease of the misfit function.

**Regularisation**
Additional terms or constraints added to the misfit function to combat ill-posedness and non-uniqueness. Regularisation encodes prior knowledge about the expected model structure (smoothness, sparsity, sharp boundaries) and prevents the inversion from fitting noise. The regularised objective is $J_{\text{reg}} = J_{\text{data}} + \lambda\,R(\mathbf{m})$, where $\lambda$ controls the trade-off between data fit and model simplicity. Choosing $\lambda$ is a key practical challenge.

**Total Variation (TV) Regularisation**
A regularisation functional that penalises the $L_1$ norm of the model gradient: $R_{\text{TV}} = \sum_{i,j} |\nabla m_{i,j}|$. TV is unique among common regularisers in that it preserves sharp edges and discontinuities while suppressing oscillations in smooth regions. This makes it ideal for GPR inversion of concrete structures where the boundaries between concrete and rebar are genuinely discontinuous. An isotropic variant uses $\sqrt{(\partial_x m)^2 + (\partial_y m)^2}$ at each cell.

**$L_2$ Regularisation (Tikhonov)**
Regularisation that penalises the squared $L_2$ norm of the model or its gradient: $R = \|\mathbf{m} - \mathbf{m}_{\text{ref}}\|_2^2$. It promotes smooth models close to a reference model. While simple and well-understood, $L_2$ regularisation tends to blur sharp boundaries, making it less suitable than TV for imaging discrete objects like rebars in concrete. It is sometimes used in early iterations for stability before switching to TV.

**Cycle-Skipping**
A fundamental failure mode of waveform inversion that occurs when the predicted and observed waveforms differ by more than half a period. In this situation, the gradient points toward aligning the wrong cycle of the waveform, driving the inversion to a local minimum far from the true model. Mitigation strategies include starting with low frequencies (multi-scale approach), using a good initial model, or employing alternative misfit functions (envelope, optimal transport) that have wider basins of attraction.

**Local Minimum**
A point in model space where the misfit function is lower than at all nearby points but not globally minimum. Gradient-based methods like L-BFGS-B can only find local minima, so the final result depends on the starting model. For FWI, this means that a poor initial guess (too far from the truth) can lead to a geologically meaningless result. Using a reasonable starting model (e.g., homogeneous concrete with known average $\varepsilon_r$) helps the inversion converge to the correct basin.

**Multi-Scale Inversion**
A strategy to mitigate cycle-skipping by starting the inversion with low-frequency data (which has wider basins of attraction and smoother misfit landscapes) and progressively adding higher frequencies. At each scale, the model from the previous scale serves as the starting point. This hierarchical approach builds up the large-scale structure first and then refines the details, greatly improving convergence to the global minimum.

**Cross-Correlation (Zero-Lag)**
In the context of FWI gradient computation, the zero-lag temporal cross-correlation between the forward wavefield and the adjoint wavefield at each grid point. This operation produces the sensitivity kernel: locations where both fields are large and temporally aligned receive large gradient values, indicating strong sensitivity. The cross-correlation is accumulated time-step by time-step during the adjoint simulation, requiring access to the stored forward field.

---

### Materials and Media

**PEC (Perfect Electric Conductor)**
An idealised material with infinite conductivity in which the tangential electric field is exactly zero ($\mathbf{n} \times \mathbf{E} = 0$). In FDTD, PEC is trivially implemented by setting $E$-field components inside the conductor to zero at every time step. Steel rebars are modelled as PEC because their conductivity ($\sim\!10^7$ S/m) is so high that essentially no field penetrates at GPR frequencies. PEC produces total reflection with a $180°$ phase reversal.

**Dielectric**
A non-conducting or weakly conducting material characterised primarily by its permittivity. In GPR work, most subsurface materials (concrete, soil, rock, ice) are treated as lossy dielectrics — they have a permittivity that determines wave speed and a small conductivity that causes attenuation. A perfect (lossless) dielectric has $\sigma = 0$ and supports wave propagation without any energy loss.

**Dispersive Medium**
A material whose electromagnetic properties (permittivity, conductivity) vary with frequency. Most real materials are dispersive to some degree; concrete's permittivity decreases with increasing frequency across the GPR band. In FDTD, frequency-dependent behaviour cannot be handled by simple scalar constants and requires auxiliary techniques such as ADE (Auxiliary Differential Equations) or recursive convolution. Ignoring dispersion in strongly dispersive media leads to incorrect waveform shapes and arrival times.

**Debye Model**
A single-pole relaxation model for frequency-dependent permittivity: $\varepsilon(\omega) = \varepsilon_\infty + (\varepsilon_s - \varepsilon_\infty)/(1 + j\omega\tau)$, where $\varepsilon_s$ is the static permittivity, $\varepsilon_\infty$ is the high-frequency limit, and $\tau$ is the relaxation time. In FDTD it is implemented via an auxiliary differential equation that adds one extra variable per cell. A single Debye pole is often sufficient to model concrete dispersion across the 0.5-3 GHz GPR band. Water is well described by a Debye model with $\tau \approx 8$ ps.

**Cole-Cole Model**
A generalisation of the Debye model that introduces a fractional exponent to broaden the relaxation spectrum: $\varepsilon(\omega) = \varepsilon_\infty + (\varepsilon_s - \varepsilon_\infty)/(1 + (j\omega\tau)^{1-\alpha})$, where $0 < \alpha < 1$. It better fits materials with a distribution of relaxation times (e.g., wet soils, biological tissues). Implementation in FDTD is more complex than Debye, typically requiring multiple auxiliary variables or Pade approximations.

**Loss Tangent ($\tan\delta$)**
The ratio of the imaginary to the real part of the complex permittivity: $\tan\delta = \varepsilon'' / \varepsilon' = \sigma / (\omega\varepsilon)$. It quantifies how "lossy" a material is at a given frequency — a high loss tangent means strong attenuation. For dry concrete, $\tan\delta \approx 0.01$-$0.05$; for wet clay, it can exceed 1. Materials with $\tan\delta > 0.5$ are generally opaque to GPR. The loss tangent is frequency-dependent, which is one reason GPR penetration varies with centre frequency.

**ADE (Auxiliary Differential Equations)**
A method to incorporate frequency-dependent material properties into FDTD by introducing auxiliary variables that evolve alongside the main electromagnetic fields. Instead of performing convolutions in the frequency domain, ADE adds one or more ordinary differential equations per cell that track the polarisation current. Each Debye pole requires one auxiliary variable; Cole-Cole models may require several. ADE is the standard approach for dispersive FDTD because it preserves the explicit time-stepping structure.

---

### Boundary Conditions

**PML (Perfectly Matched Layer)**
An absorbing boundary condition that terminates the FDTD computational domain by surrounding it with a layer of artificial material designed to absorb outgoing waves with zero (in theory) reflection. Introduced by Berenger in 1994, PML works by applying a complex coordinate stretching that causes waves to decay exponentially as they propagate into the layer. In practice, discretisation introduces small reflections, so PML layers are typically 10-20 cells thick with a graded conductivity profile. PML is the gold standard for FDTD domain termination.

**CPML (Convolutional Perfectly Matched Layer)**
A recursive-convolution implementation of PML that is algebraically equivalent to CFS-PML but more efficient to code. Instead of splitting field components (as in Berenger's original formulation), CPML uses auxiliary "psi" variables that accumulate the convolution integrals via a simple two-term recursion at each time step. This unsplit formulation is easier to implement, more memory-efficient, and numerically identical to the split-field version. CPML is the most widely used PML variant in modern FDTD codes.

**CFS-PML (Complex Frequency Shifted PML)**
An enhanced PML formulation that generalises the standard PML by introducing three parameters in the complex coordinate stretching: a conductivity-like term $\sigma$ for absorption, a real stretching factor $\kappa$ for improved grazing-incidence performance, and a frequency-shift parameter $\alpha$ that stabilises the absorption of evanescent (non-propagating) waves. Standard PML can amplify evanescent waves and cause late-time instability; the $\alpha$ parameter in CFS-PML prevents this by shifting the pole away from zero frequency. Typical values are $\kappa_{\max} = 5$-$15$, $\sigma_{\max}$ chosen by an optimal formula, and $\alpha_{\max} = 0.02$-$0.05$.

**Absorbing Boundary Condition (ABC)**
A general term for any technique used to truncate the infinite physical domain to a finite computational domain without introducing spurious reflections. ABCs predate PML and include analytical approaches like the Mur first- and second-order conditions, which apply approximate one-way wave equations at the boundary. While simpler than PML, these classical ABCs offer much poorer absorption (especially at non-normal incidence angles) and have been largely superseded by PML in modern FDTD practice.

**Mur ABC**
A simple absorbing boundary condition derived from a one-way wave equation approximation. The first-order Mur ABC is exact only for normal incidence and introduces significant reflection at oblique angles. The second-order Mur ABC improves this but is still far less effective than PML, especially for broadband sources. Mur conditions remain useful as a lightweight fallback when PML's memory overhead is prohibitive or for quick prototyping.

**PML Grading (Polynomial Profile)**
The spatial variation of the PML absorption parameter $\sigma$ from zero at the inner boundary to $\sigma_{\max}$ at the outer boundary. A gradual increase (typically polynomial of order 3-4: $\sigma(d) = \sigma_{\max}(d/L)^n$) is essential because an abrupt jump in $\sigma$ would cause reflections at the PML interface. The optimal $\sigma_{\max}$ depends on PML thickness and polynomial order; values too low give insufficient absorption, while values too high cause discretisation-induced reflections.

---

### Source and Signal Concepts

**Ricker Wavelet**
The second derivative of a Gaussian function, also known as the "Mexican hat" wavelet. It is the most common source waveform in GPR and seismic modelling because it is compact in both time and frequency, has zero DC component (no static offset), and is defined by a single parameter: the peak frequency $f_p$. Its bandwidth extends from approximately $0.2\,f_p$ to $3\,f_p$, with the -6 dB bandwidth spanning roughly $0.6\,f_p$ to $1.8\,f_p$. In this project, $f_p = 1.5$ GHz gives energy from about 0.3 to 4.5 GHz.

**Centre Frequency (Peak Frequency)**
The frequency at which the source wavelet's amplitude spectrum reaches its maximum. For a Ricker wavelet, this equals the parameter $f_p$ used in its mathematical definition. The centre frequency is a primary design parameter: it controls the trade-off between resolution (higher frequency = shorter wavelength = finer detail) and penetration depth (higher frequency = more attenuation). GPR systems for concrete inspection typically use 1-2.5 GHz.

**Bandwidth**
The range of frequencies over which the source wavelet carries significant energy, typically defined at the -3 dB or -6 dB level. Broader bandwidth means shorter pulses in the time domain, which improves temporal (and hence depth) resolution. In FWI, the bandwidth determines what spatial scales of the model can be recovered: low frequencies constrain the large-scale structure, while high frequencies provide the fine detail. A Ricker wavelet with $f_p = 1.5$ GHz has a usable bandwidth of roughly 0.3 to 4.5 GHz.

**Soft Source**
A source implementation in FDTD where the source waveform is added to the existing field value: $E_z^{n+1}(i_s, j_s) \mathrel{+}= S(t)$. This is equivalent to injecting a current density $J_z$ at the source location. The key advantage is that waves can pass through the source point without reflection — the source is "transparent" to the wavefield. This is essential for GPR modelling where the source is inside the domain and must not create artificial scattering.

**Hard Source**
A source implementation where the field value is set (overwritten) directly: $E_z^{n+1}(i_s, j_s) = S(t)$. This creates an artificial perfect electric conductor at the source location after the pulse ends (when $S(t) = 0$, the field is forced to zero). Any wavefield arriving at the source point reflects off it, contaminating the simulation with spurious arrivals. Hard sources are appropriate only when the source is on a domain boundary, never for interior sources in GPR modelling.

**DC Component**
The zero-frequency content of a signal. A source wavelet with a non-zero DC component would produce a static electric field that never decays, causing the simulation to drift and PML to become ineffective (PML cannot absorb static fields). The Ricker wavelet is naturally DC-free because it is the second derivative of a Gaussian. Ensuring zero DC content is critical for FDTD stability and for obtaining a clean, physically meaningful B-scan.

**Current Density Source ($J_z$)**
The physical quantity injected into the $E_z$ update equation to model a radiating antenna in FDTD. Ampere's law includes the term $\mathbf{J}$: $\nabla \times \mathbf{H} = \varepsilon\,\partial\mathbf{E}/\partial t + \sigma\mathbf{E} + \mathbf{J}$. Discretising this and rearranging gives the additive soft-source term. The current density source radiates energy into all directions equally (for a point source), which is a reasonable first approximation to a GPR antenna's near-field behaviour.

**Point Source**
A source occupying a single grid cell, representing an idealised omnidirectional radiator. While real GPR antennas have finite physical extent and directional radiation patterns, a point source captures the essential physics and is standard in 2D FDTD modelling. The point-source approximation breaks down when the antenna size is comparable to the wavelength or when near-field antenna effects significantly influence the recorded data.

---

### Key Physical Constants and Parameters

**Vacuum Permittivity ($\varepsilon_0$)**
The fundamental constant $\varepsilon_0 \approx 8.854 \times 10^{-12}$ F/m that appears in Coulomb's law and Maxwell's equations. It sets the scale for electric field propagation in free space. In the FDTD update equations and in the adjoint gradient formula, $\varepsilon_0$ appears as a multiplicative factor. It also defines the baseline for relative permittivity: $\varepsilon = \varepsilon_r \cdot \varepsilon_0$.

**Vacuum Permeability ($\mu_0$)**
The fundamental constant $\mu_0 = 4\pi \times 10^{-7}$ H/m governing magnetic field behaviour in free space. Together with $\varepsilon_0$, it determines the speed of light: $c = 1/\sqrt{\mu_0 \varepsilon_0} \approx 3 \times 10^8$ m/s. In GPR applications, $\mu_0$ is the only permeability value used because all relevant materials are non-magnetic ($\mu_r = 1$).

**Speed of Light ($c$)**
The speed of electromagnetic wave propagation in vacuum, $c \approx 3 \times 10^8$ m/s. In a dielectric medium with relative permittivity $\varepsilon_r$, the wave speed reduces to $v = c / \sqrt{\varepsilon_r}$. For concrete ($\varepsilon_r \approx 6$), $v \approx 0.122$ m/ns. The speed of light in the fastest medium (usually air, $\varepsilon_r = 1$) determines the CFL time-step limit for the FDTD simulation.